In [ ]:
import spacy
import scispacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector

In [ ]:
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe("scispacy_linker", config={"linker_name": "umls"})
nlp.add_pipe("abbreviation_detector")

c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/mod

In [ ]:
text = """
Medications:
Mycophenolate mofetil (MMF)

The patient has lupus nephritis and was prescribed rituximab and MMF.
Proteinuria has improved following treatment.
"""

doc = nlp(text)

In [32]:
abbreviation_map = {
    str(abbr): str(abbr._.long_form)
    for abbr in doc._.abbreviations
}

print(abbreviation_map)

{'MMF': 'Mycophenolate mofetil'}


In [37]:
linker = nlp.get_pipe("scispacy_linker")

for entity in doc.ents:
    entity_text = entity.text

    normalised_text = abbreviation_map.get(entity_text, entity_text)

    normalised_doc = nlp(normalised_text)

    if normalised_doc.ents:
        normalised_entity = normalised_doc.ents[0]

        if normalised_entity._.kb_ents:
            cui, score = normalised_entity._.kb_ents[0]
            concept = linker.kb.cui_to_entity[cui]

            print("Original:", entity.text)
            print("Normalised:", normalised_text)
            print("Canonical:", concept.canonical_name)
            print("Semantic types:", concept.types)
            print("Score:", score)
            print()

Original: Medications
Normalised: Medications
Canonical: Pharmaceutical Preparations
Semantic types: ['T121']
Score: 0.9835557341575623

Original: MMF
Normalised: Mycophenolate mofetil
Canonical: mycophenolate mofetil
Semantic types: ['T109', 'T121']
Score: 0.9700199961662292

Original: patient
Normalised: patient
Canonical: Patients
Semantic types: ['T101']
Score: 0.9829968214035034

Original: lupus nephritis
Normalised: lupus nephritis
Canonical: Lupus Nephritis
Semantic types: ['T047']
Score: 0.9638109803199768

Original: prescribed
Normalised: prescribed
Canonical: Prescribed
Semantic types: ['T058']
Score: 0.9690635204315186

Original: rituximab
Normalised: rituximab
Canonical: rituximab
Semantic types: ['T116', 'T121', 'T129']
Score: 0.9763724207878113

Original: MMF
Normalised: Mycophenolate mofetil
Canonical: mycophenolate mofetil
Semantic types: ['T109', 'T121']
Score: 0.9700199961662292

Original: Proteinuria
Normalised: Proteinuria
Canonical: Proteinuria
Semantic types: ['T0